# 02 — Train RG-GeoPrompt-PEFT on Potsdam

**Purpose:** train the cosine-head GeoPrompt model (init from the
DINOv2+LoRA 84.6% checkpoint), then evaluate **mIoU (5 cls)** and
**F1 (6 cls incl. clutter)** for all available models.

**Needs:** Kaggle **T4 GPU**, Potsdam patch dataset, `HF_TOKEN` Kaggle
Secret, `rg-geoprompt-src` dataset attached.

**Outputs:** `geoprompt_best.pth`, `geoprompt_ckpt_epoch*.pth`,
`geoprompt_training_log.csv` in `/kaggle/working`, backed up to HF.

**Already done — never retrain:** SegFormer (82.1%), DINOv2+LoRA (84.6%).

In [ ]:
# Bootstrap: clone repo and add src/ to sys.path. Run every session.
import subprocess, sys
from pathlib import Path

if Path("/kaggle/input").exists():                 # Kaggle
    CLONE_DIR = Path("/kaggle/working/VC/rg-geoprompt-peft")
    if not CLONE_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/HarishDeepak/VC.git", str(CLONE_DIR)],
            check=True)
        print(f"✓ cloned → {CLONE_DIR}")
    else:
        print(f"✓ repo present → {CLONE_DIR}")
    sys.path.insert(0, str(CLONE_DIR / "src"))
else:                                              # local (VS Code)
    for cand in ["../src", "src"]:
        if Path(cand, "rg_geoprompt").exists():
            sys.path.insert(0, str(Path(cand).resolve()))
            break

from rg_geoprompt import paths
print(paths.describe())

In [ ]:
# Installs (idempotent, ~1 min). Run every session.
%pip install -q peft open_clip_torch
print("✓ peft + open_clip_torch installed")

## Data — tile-level split (random split is FORBIDDEN)

In [ ]:
from rg_geoprompt.datasets import build_potsdam_loaders
train_loader, val_loader = build_potsdam_loaders()
print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)}")
imgs, lbls = next(iter(train_loader))
print(f"batch: {imgs.shape} {imgs.dtype} | labels: {lbls.shape} {lbls.dtype}")

## Text prototypes — reuse `text_embeddings.pt` if it exists
CLIP only runs when the file is missing (pull it from HF first if needed).

In [ ]:
from rg_geoprompt import paths
from rg_geoprompt.utils import ensure_checkpoint
from rg_geoprompt.prompts import load_or_encode_text_embeddings

# Try to restore the cached embeddings from HF before re-encoding
if not paths.TEXT_EMBEDDINGS_PT.exists():
    cached = ensure_checkpoint("text_embeddings.pt", required=False)
    if cached and cached != paths.TEXT_EMBEDDINGS_PT:
        import shutil; shutil.copy(cached, paths.TEXT_EMBEDDINGS_PT)

text_embeddings = load_or_encode_text_embeddings()
print("prototypes:", tuple(text_embeddings.shape))   # (6, 512)

## Build GeoPrompt model from the DINOv2+LoRA checkpoint

In [ ]:
# REQUIRES: Kaggle T4 GPU
import torch
from rg_geoprompt import paths
from rg_geoprompt.constants import DEVICE
from rg_geoprompt.models_geoprompt import build_geoprompt_from_dinov2_ckpt
from rg_geoprompt.utils import ensure_checkpoint, set_seed

set_seed(42)
dinov2_ckpt = ensure_checkpoint("dinov2_lora_best.pth")   # local or HF
geo_model = build_geoprompt_from_dinov2_ckpt(text_embeddings, dinov2_ckpt)

total = sum(p.numel() for p in geo_model.parameters())
trainable = sum(p.numel() for p in geo_model.parameters() if p.requires_grad)
print(f"total {total:,} | trainable {trainable:,} ({100*trainable/total:.2f}%)")
print(f"τ init: {geo_model.tau.item():.4f}")
with torch.no_grad():
    out = geo_model(torch.randn(1, 3, 512, 512, device=DEVICE))
print("forward OK:", tuple(out.shape))   # (1, 6, 512, 512)
# If this cell or training OOMs: rebuild with lowres_similarity=True
# (see models_geoprompt.py docstring) and note the change in the report.

## Training — 10 epochs, LR=1e-4, append-safe log, epoch guards

In [ ]:
# REQUIRES: Kaggle T4 GPU  (~45 min)
import torch, torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from rg_geoprompt import paths
from rg_geoprompt.constants import DEVICE, EVAL_EVERY, GEO_EPOCHS, GEO_LR
from rg_geoprompt.losses import seg_loss
from rg_geoprompt.metrics import compute_miou
from rg_geoprompt.utils import append_log_row

import csv as _csv
if paths.GEOPROMPT_LOG_CSV.exists():
    _rows = list(_csv.DictReader(open(paths.GEOPROMPT_LOG_CSV)))
    START_EPOCH = int(_rows[-1]["epoch"]) if _rows else 0
    best_miou = max((float(r["miou"])/100 for r in _rows if r["miou"]), default=0.0)
    best_epoch = START_EPOCH
    print(f"Resuming from epoch {START_EPOCH}, best mIoU so far: {best_miou*100:.1f}%")
else:
    START_EPOCH, best_miou, best_epoch = 0, 0.0, 0
    print("Fresh training run (no log found).")

optimizer = AdamW((p for p in geo_model.parameters() if p.requires_grad),
                  lr=GEO_LR, weight_decay=0.01)
scheduler = OneCycleLR(optimizer, max_lr=GEO_LR,
                       steps_per_epoch=len(train_loader),
                       epochs=GEO_EPOCHS, pct_start=0.05)

for epoch in range(START_EPOCH, GEO_EPOCHS):
    geo_model.train()
    tr = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        loss = seg_loss(geo_model(imgs), lbls)
        loss.backward(); optimizer.step(); scheduler.step()
        tr += loss.item()
    tr /= len(train_loader)

    geo_model.eval(); vl = 0.0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            vl += seg_loss(geo_model(imgs), lbls).item()
    vl /= len(val_loader)

    miou_val = 0.0
    if (epoch + 1) % EVAL_EVERY == 0:
        _, miou_val = compute_miou(geo_model, val_loader)
        if miou_val > best_miou:
            best_miou, best_epoch = miou_val, epoch + 1
            torch.save(geo_model.state_dict(), paths.GEOPROMPT_CKPT)
        torch.save({"epoch": epoch + 1,
                    "model_state_dict": geo_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_miou": best_miou},
                   paths.WORK_DIR / f"geoprompt_ckpt_epoch{epoch+1}.pth")

    tau = geo_model.tau.item()
    print(f"Epoch {epoch+1:02d}/{GEO_EPOCHS}  train={tr:.4f}  val={vl:.4f}"
          f"  τ={tau:.4f}" + (f"  mIoU={miou_val*100:.1f}%"
                              if (epoch + 1) % EVAL_EVERY == 0 else ""))
    append_log_row(paths.GEOPROMPT_LOG_CSV,
                   ["epoch", "train_loss", "val_loss", "miou", "tau"],
                   [epoch + 1, round(tr, 4), round(vl, 4),
                    round(miou_val * 100, 2) if (epoch + 1) % EVAL_EVERY == 0 else "",
                    round(tau, 4)])

print(f"\nBest GeoPrompt mIoU: {best_miou*100:.1f}% @ epoch {best_epoch}")
print("DINOv2+LoRA baseline : 84.6% | SegFormer baseline: 82.1%")
print("Sanity: τ should drift DOWN from 0.07 toward ~0.04 over training.")

## Evaluation — mIoU (5 cls) and F1 (6 cls) — never swap the two

In [ ]:
# REQUIRES: Kaggle T4 GPU
from rg_geoprompt.metrics import compute_f1, compute_miou
from rg_geoprompt.constants import CLASS_NAMES
from rg_geoprompt.utils import print_f1_report

iou, miou = compute_miou(geo_model, val_loader)   # 5 cls, ignore (5, 255)
print("Per-class IoU (val tile 6_15):")
for i, name in enumerate(CLASS_NAMES[:5]):
    print(f"  [{i}] {name:12s}: {iou[i]:.4f}")
print(f"mIoU (5 classes, clutter excluded): {miou*100:.2f}%")

f1_geo = compute_f1(geo_model, val_loader)        # 6 cls, ignore (255,)
print_f1_report(f1_geo, "RG-GeoPrompt-PEFT")

## F1 for the baselines (Table 1 of the report needs all three)

In [ ]:
# REQUIRES: Kaggle T4 GPU
from rg_geoprompt.models_dino_lora import load_dinov2_lora
from rg_geoprompt.metrics import compute_f1, segformer_logits_fn
from rg_geoprompt.utils import ensure_checkpoint, print_f1_report

dino_model = load_dinov2_lora(ensure_checkpoint("dinov2_lora_best.pth"))
print_f1_report(compute_f1(dino_model, val_loader), "DINOv2 + LoRA")
del dino_model; import torch; torch.cuda.empty_cache()

# SegFormer (optional — needs transformers + its checkpoint):
# from transformers import SegformerForSemanticSegmentation
# seg = SegformerForSemanticSegmentation.from_pretrained(
#     "nvidia/mit-b0", num_labels=6).to("cuda")
# state = torch.load(ensure_checkpoint("segformer_b0_potsdam_baseline.pth"),
#                    map_location="cuda")
# seg.load_state_dict(state)
# print_f1_report(compute_f1(seg, val_loader, logits_fn=segformer_logits_fn),
#                 "SegFormer-B0")

## Backup to HuggingFace (token from Kaggle Secrets — never hardcoded)

In [ ]:
from rg_geoprompt.utils import hf_backup
hf_backup()   # uploads checkpoints, logs and text_embeddings.pt

In [ ]:
# ═══ CELL-END CHECKLIST — run before closing the session ═══
from rg_geoprompt import paths
checks = {
    "geoprompt_best.pth saved": paths.GEOPROMPT_CKPT.exists(),
    "training log exists": paths.GEOPROMPT_LOG_CSV.exists(),
    "text_embeddings.pt exists": paths.TEXT_EMBEDDINGS_PT.exists(),
    "dinov2_lora_best.pth present": paths.DINOV2_LORA_CKPT.exists(),
}
for k, v in checks.items():
    print(("✓" if v else "✗"), k)
print("\nIf any ✗ above is unexpected, fix it BEFORE the HF backup cell.")
print("Then: 1) run the HF backup cell  2) optionally download .pth files")
print("from the Kaggle Output tab as the third backup layer.")